# Load Libraries

In [14]:


%load_ext autoreload
%autoreload 2



# Basic libraries
import pandas as pd
import numpy as np
import sys



# Scipy
import scipy
from scipy import signal
from scipy.linalg import solve
from scipy import constants
from scipy.interpolate import interp1d
from scipy.fft import fft, fftfreq

# Locate files
import os
from pathlib import Path
from glob import glob

# Plots
from matplotlib import pyplot as plt

# Define project root (run this notebook from MHDTurbPy repository root)
root_dir = str(Path.cwd())



# Make sure to use the local spedas
sys.path.insert(0, str(Path(root_dir) / 'pyspedas'))
import pyspedas
from pyspedas.utilities import time_string
from pytplot import get_data
from joblib import Parallel, delayed

""" Import manual functions """

sys.path.insert(1, str(Path(root_dir) / 'functions'))
import calc_diagnostics as calc
import TurbPy as turb
import general_functions as func
import Figures as figs
import interactive_figs


from   SEA import SEA
import three_D_funcs as threeD
import download_data as download

sys.path.insert(1, os.path.join(os.getcwd(), 'functions/3d_anis_analysis_toolboox'))
import collect_wave_coeffs 
import data_analysis 



os.environ["CDF_LIB"] = "/Applications/cdf/cdf/lib"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
"""
mre_run_solo.py

Minimal-but-complete MRE to run the SOLO pipeline.

RUN FROM REPO ROOT (the directory that contains: pyspedas/, functions/, download_data.py)
"""

from __future__ import annotations

import os
from pathlib import Path
from joblib import Parallel, delayed

import download_data as download


# ---------------------------------------------------------------------
# CDFlib path (needed for SOLO distance download via CDAS)
# ---------------------------------------------------------------------
cdf_lib_path = "/Applications/cdf/cdf/lib"


# ---------------------------------------------------------------------
# Credentials (None is fine for SOLO in most cases)
# ---------------------------------------------------------------------
credentials = None


# ---------------------------------------------------------------------
# Settings (organized by topic, then flattened into one dict)
# ---------------------------------------------------------------------
Paths = {
    
'Data_path'              : 'C:\\Users\\nokni\\work\\Data/',  # Path were pyspedas downloads data
'save_destination'      : 'C:\\Users\\nokni\\work\\MHDTurbPy\\examples\\',
    "SOLO_dist_path"    : "/Users/turbulator/work/turb_amplitudes/SOLO/solo_dist.pkl",
}

Mission = { 
    "sc"             : "SOLO",
    "in_rtn"         : 0,
    "use_local_data" : False,
}

Intervals = {
    # "start_date": "2025-01-10 00:00",
    # "end_date": "2025-01-10 06:00",

    "start_date": "2022-10-01 00:00",
    "end_date": "2022-10-03 00:00",
    
    "multiple_intervals": False,   # False -> single interval, True -> use Step/duration below
    "duration": "4H",
    "Step": "210min",
    "addit_time_around": 1,        # hours padding in LoadTimeSeriesSOLO
}

Sampling = {
    "part_resol": 1000,
    "MAG_resol": 1000,
    "upsample_low_freq_ts": False,
}

QualityControl = {
    "overwrite_files"   :  1,
    "must_have_qtn": False,
    "Max_par_missing": 30,
    "gap_time_threshold": 5,
    "use_hampel": False,
    "hampel_params": {"w": 200, "std": 3},
}

Output = {
    
    "save_all": True,
    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

Diagnostics = {
    "estimate_derived_param": True,
    "rol_window": "60min",

    "PSDs": {"flag": False},
    "struc_funcs": {"flag": False},

    "npt_struc_funcs": {
        "flag": False,
        "five_points_sfunc": True,
        "return_Bmod": True,
        "dt_step": 0.25,
        "est_sfuncs": False,
        "max_qorder": 8,
    },

    "estimate_psd_b": True,
    "estimate_psd_v": True,
    "est_PSD_components": True,
    "smooth_psd": False,
}

Gaps = {
    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 500,
        "Par_big_gaps": 500,
        "QTN_big_gaps": 10,
    }
}

SOLO = {
    "SOLO_use_merged_MAG": False,   # set False to use SPEDAS L2 MAG
    "SOLO_merged_fs": 256,         # 256 or 4096
}

# ---- Unified MAG noise removal (works for SOLO merged MAG and PSP SCAM)
# This is the preferred key; legacy Mag_SCAM_PSP["noise_flag"] still works.
MAG_NOISE = {
    "Mag_SCM": {
        "noise_flag": False,
        "noise_removal": {
            "freq_min": 8.0,

            "stft_nperseg": 2048,
            "stft_overlap": 0.5,

            "percentile_q": 99.5,
            "kernel": 301,
            "thresh_db": 3.0,
            "merge_hz": 0.20,
            "max_lines": 2000,

            "track_half_width_hz": 5.0,
            "remove_half_width_hz": 1.5,
            "atten_db": 100.0,

            "fallback_top_k": 120,
            "whiten_exp": 0.0,
        },
    }
}







# ---- final settings dict (what pipeline consumes)
settings = {
    **Paths,
    **Mission,
    **Intervals,
    **Sampling,
    **QualityControl,
    **Output,
    **Diagnostics,
    **SOLO,
    **MAG_NOISE,
    "Big_Gaps": Gaps["Big_Gaps"],
}


# ---------------------------------------------------------------------
# What variables to download (None -> defaults inside SOLO.py)
# ---------------------------------------------------------------------
vars_2_downnload = {
    "mag": None,
    "swa": None,
    "rpw": None,
    "ephem": None,
}


# ---------------------------------------------------------------------
# Interval generation
# ---------------------------------------------------------------------
generated_interval_list = download.generate_intervals(
    settings["start_date"],
    settings["end_date"],
    settings["multiple_intervals"],
    data_path=settings["Data_path"],
    settings=settings,
)


# ---------------------------------------------------------------------
# Run download for each interval
# ---------------------------------------------------------------------
os.chdir(settings["Data_path"])

save_path = Path(settings["save_destination"]).joinpath(settings["sc"])
save_path.mkdir(parents=True, exist_ok=True)

Parallel(n_jobs=1)(
    delayed(download.download_files)(
        jj,
        generated_interval_list,
        settings,
        vars_2_downnload,
        cdf_lib_path,
        credentials,
        save_path,
    )
    for jj in range(len(generated_interval_list))
)


10-Feb-26 03:00:51: Generating only one interval based on the provided start and end times.
10-Feb-26 03:00:51: Start Time: 2022-10-01 00:00:00
End Time: 2022-10-03 00:00:0000

10-Feb-26 03:00:51: Considering a single interval spanning: 2022-10-01 00:00:00 to 2022-10-03 00:00:00
Creating new folder  C:\Users\nokni\work\MHDTurbPy\examples\SOLO\2022-10-01_00-00-00_2022-10-03_00-00-00_sc_0
10-Feb-26 03:00:57: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/solar-orbiter/rpw/science/l3/bia-density-10-seconds/2022/
8: Downloading https://spdf.gsfc.nasa.gov/pub/data/solar-orbiter/rpw/science/l3/bia-density-10-seconds/2022/solo_l3_rpw-bia-density-10-seconds_20220930_v01.cdf to solar_orbiter_data/rpw/science/l3/bia-density-10-seconds/2022/solo_l3_rpw-bia-density-10-seconds_20220930_v01.cdf
10-Feb-26 03:00:59: Download complete: solar_orbiter_data/rpw/science/l3/bia-density-10-seconds/2022/solo_l3_rpw-bia-density-10-seconds_20220930_v01.cdfv01.cdf to solar_orbiter_data/rpw/science

Index(['Br', 'Bt', 'Bn', 'np', 'Vr', 'Vt', 'Vn', 'Tp', 'Vth', 'np_qtn',
       'ne_qtn', 'Dist_au', 'Br_mean', 'Bt_mean', 'Bn_mean', 'Vr_mean',
       'Vt_mean', 'Vn_mean', 'np_mean'],
      dtype='object')
0 out of 1 finished


[None]

In [2]:
#  User defined parameters
sc          = 'SOLO'
which_int   = 0
load_path   = Path(root_dir).joinpath('examples', sc)


# Locate the downloaded files
finnames      = func.load_files(load_path, 'final.pkl')
gennames      = func.load_files(load_path, 'general.pkl')
signames      = func.load_files(load_path, 'sig_c_sig_r.pkl')
maggaps       = func.load_files(load_path, 'mag_gaps.pkl')
qtngaps       = func.load_files(load_path, 'qtn_gaps.pkl')
pargaps       = func.load_files(load_path, 'par_gaps.pkl')
sc_pot        = func.load_files(load_path, 'sc_pot_gaps.pkl')


#Load the filed
fin         = pd.read_pickle(finnames[which_int])
gen         = pd.read_pickle(gennames[which_int])
sig         = pd.read_pickle(signames[which_int])
mag_gaps    = pd.read_pickle(maggaps[which_int])
qtn_gaps    = pd.read_pickle(qtngaps[which_int])
par_gaps    = pd.read_pickle(pargaps[which_int])
sc_pot_gaps = pd.read_pickle(sc_pot[which_int])

finnames[which_int]

C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\final.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\general.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\sig_c_sig_r.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\mag_gaps.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\qtn_gaps.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\par_gaps.pkl
C:\Users\nokni\work\MHDTurbPy\examples\SOLO\*\sc_pot_gaps.pkl


'C:\\Users\\nokni\\work\\MHDTurbPy\\examples\\SOLO\\2024-10-10_00-00-00_2024-10-10_06-00-00_sc_0\\final.pkl'

In [32]:
import plotly.io as pio
print("plotly renderer =", pio.renderers.default)


plotly renderer = plotly_mimetype


In [31]:
# user defined parameters

label_size        = 21                                    # labels etc

n_subplots        = 7                                     # number os subplots
my_dir            = '/Users/nokni/work/MHDTurbPy/examples/' #
format_2_return   = "%Y_%m_%d"                            # Format to save figures


figs.visualize_downloaded_intervals(
                                  sc                         ,
                                  fin['Par']['V_resampled'],
                                  fin['Mag']['B_resampled'],
                                  sig     ,
                                  my_dir,
                                  format_2_return  = "%Y_%m_%d",  #
                                  size             = label_size,
                                  numb_subplots    = n_subplots
                                 )



# Run with interactive figure

In [5]:
%matplotlib tk

from pathlib import Path
import matplotlib.pyplot as plt
import interactive_figs
import general_functions as func

plt.close("all")

sc = "SOLO"
which_int = 0
load_path = r"C:\Users\nokni\work\MHDTurbPy\examples\SOLO"
my_dir   = r"C:\Users\nokni\work\MHDTurbPy\examples"
save_path = Path(my_dir) / "selected_intervals"

# ============================================================
# ONE run that does BOTH:
# (A) add one extra curve to an existing panel (panel 0)
# (B) append a brand-new extra panel (8th subplot)
# ============================================================
panel_edits = {
    # (A) add one extra curve to an existing panel
    "add_series": {
        "panel_idx": 0,   # top panel
        "axis_idx": 0,    # first axis-spec in that panel (left axis)
        "series": {
            "kind": "col",
            "col": "B_RTN",
            "label": r"$|B|$ (extra dashed)",
            "style": {"lw": 1.0, "ls": "--", "ms": 0, "color": "0.35"},
        },
    },

    # (B) append a new panel at the bottom
    "add_panels": {
        "where": "append",
        "panel": {
            "axes": [
                {
                    "axis_id": "left",
                    "source": "Par",
                    "scale": "linear",
                    "series": [
                        {
                            "kind": "col",
                            "col": "Vth",
                            "label": r"$T_p~[eV]$ (extra panel)",
                            "style": {"lw": 0.8, "ls": "-", "ms": 0, "color": "k"},
                        }
                    ],
                    "legend": {"fontsize": "small", "frameon": False, "bbox_to_anchor": (1.01, 1), "loc": 2},
                    "hline": [{"y": 100.0, "ls": ":", "c": "0.3", "lw": 1.2}],
                }
            ]
        },
    },

    # (c) More

    
}

fig, events = interactive_figs.interactive_mhdturbpy_interval(
    sc=sc,
    which_int=which_int,
    load_path=load_path,
    my_dir=my_dir,
    save_path=save_path,
    rolling=None,
    resample_rule=None,
    fill_method=None,
    gap_thresholds={"mag": "30s", "par": "30s", "qtn": "30s", "sc_pot": "30s"},
    load_files_func=func.load_files,
    autosave=True,
    resume=True,
    export_csv=True,
    snap_to_data=False,
    enable_comments=True,
    debug_interaction=True,
    panel_config=None,          # use defaults
    panel_edits=panel_edits,    # add curve + append panel
    auto_ylims=True,
    debug_plot_config=True,     # should print n_panels=8
)

plt.show()


[plot] n_panels=8
[picker] resumed 3 intervals from C:\Users\nokni\work\MHDTurbPy\examples\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl
installed mpl(cids=16,None,17,18) + tkbinds=3MHDTurbPy\examples\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl

[picker] left_select_fallback(default)=True  (toggle with 'l')s\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl


dedupe_ms=250  dedupe_tol_ns=5000000  span_axes=8 marker_axes=10  move_throttle_ms=33  use_release_event=False0_10_SOLO.pkl



